# Breast Tumour — Combination C Ensemble Pipeline Notebook

**Ensemble:** MONAI ResNet UNet 3D + DynUNet 3D + 2D ResNet34 U-Net  
**Data:** `data/patients_combined/` (233 patients, 1.5mm isotropic)  
**Best combo found:** MONAI=0.30 · DynUNet=0.40 · 2D=0.30 · threshold=0.50  
**Ensemble test Dice:** 0.6744 · Detection rate: 100% · Missed: 0  

```
Cell 1  — Environment check + load all three models
Cell 2  — Weight sweep visualisation (val set results)
Cell 3  — Test set inference (ensemble predictions)
Cell 4  — GT vs prediction overlay (all 3 model probs + ensemble)
Cell 5  — 3D MIP rendering: GT vs ensemble prediction
Cell 6  — Per-patient evaluation table (sortable, colour-coded)
Cell 7  — Model contribution analysis (which model helps which patient)
Cell 8  — 2D baseline vs MONAI vs DynUNet vs Ensemble comparison
Cell 9  — Full summary card
```

In [1]:
# ============================================================
# CELL 1 — Environment check + load all three models
# ============================================================
import os, sys, warnings, json
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from IPython.display import display
import pandas as pd
from scipy import ndimage
from tqdm.notebook import tqdm
from collections import Counter
warnings.filterwarnings('ignore')

# ---- GPU ----
assert torch.cuda.is_available(), 'No CUDA GPU found.'
device   = torch.device('cuda')
torch.cuda.set_device(0)
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

# ---- Project root ----
_cwd = os.getcwd()
if os.path.basename(_cwd).lower() in ('notebooks', 'notebook'):
    PROJECT_ROOT = os.path.dirname(_cwd)
elif not os.path.exists(os.path.join(_cwd, 'src', 'segmentation')):
    _parent = os.path.dirname(_cwd)
    PROJECT_ROOT = _parent if os.path.exists(os.path.join(_parent, 'src', 'segmentation')) else _cwd
else:
    PROJECT_ROOT = _cwd
os.chdir(PROJECT_ROOT)

SRC_DIR = os.path.join(PROJECT_ROOT, 'src', 'segmentation')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from data_3d       import PATCH_SIZE, _build_data_list, NpyDataset, get_val_transforms
from model_3d      import get_model  as get_monai
from model_dynunet import get_dynunet
import segmentation_models_pytorch as smp

# ---- Paths ----
PREPROCESSED_ROOT   = os.path.join(PROJECT_ROOT, 'data', 'patients_combined')
MONAI_MODEL_PATH    = os.path.join(PROJECT_ROOT, 'models', 'segmentation_3d', 'unet3d_best_raw.pth')
DYNUNET_MODEL_PATH  = os.path.join(PROJECT_ROOT, 'models', 'dynunet_3d', 'dynunet_best_raw.pth')
MODEL_2D_PATH       = os.path.join(PROJECT_ROOT, 'models', '2d_models', 'smooth_0.8085', 'unet_best.pth')
ENSEMBLE_OUT_DIR    = os.path.join(PROJECT_ROOT, 'outputs', 'ensemble_3model')
ENSEMBLE_CSV        = os.path.join(ENSEMBLE_OUT_DIR, 'ensemble_3model_evaluation.csv')
os.makedirs(ENSEMBLE_OUT_DIR, exist_ok=True)

# ---- Best combo from weight sweep ----
# Update these if you re-run ensemble_3model.py with different results
BEST_W_MONAI   = 0.30
BEST_W_DYNUNET = 0.40
BEST_W_2D      = 0.30
BEST_THRESHOLD = 0.50
BASELINE_2D    = 0.7501
MONAI_TEST     = 0.6520
DYNUNET_TEST   = 0.5188  # update after DynUNet finishes training
ENSEMBLE_TEST  = 0.6744

# ---- Load MONAI model ----
print('Loading MONAI ResNet UNet 3D...')
monai_model = get_monai(device)
monai_model.load_state_dict(torch.load(MONAI_MODEL_PATH, map_location=device, weights_only=True))
monai_model.eval()
print(f'  Loaded: {MONAI_MODEL_PATH}')

# ---- Load DynUNet ----
print('Loading DynUNet 3D...')
dyn_model = get_dynunet(device)
dyn_model.load_state_dict(torch.load(DYNUNET_MODEL_PATH, map_location=device, weights_only=True))
dyn_model.eval()
print(f'  Loaded: {DYNUNET_MODEL_PATH}')

# ---- Load 2D model ----
print('Loading 2D ResNet34 U-Net...')
model_2d = smp.Unet(encoder_name='resnet34', encoder_weights=None,
                    in_channels=3, classes=1, activation=None).to(device)
model_2d.load_state_dict(torch.load(MODEL_2D_PATH, map_location=device, weights_only=True))
model_2d.eval()
print(f'  Loaded: {MODEL_2D_PATH}')

print(f'\nAll 3 models loaded on {gpu_name} ({vram_gb:.1f} GB)')
print(f'Ensemble weights: MONAI={BEST_W_MONAI}  DynUNet={BEST_W_DYNUNET}  2D={BEST_W_2D}  thresh={BEST_THRESHOLD}')

ModuleNotFoundError: No module named 'data_3d'

In [ ]:
# ============================================================
# CELL 2 — Weight sweep visualisation
# Reads the sweep results printed by ensemble_3model.py
# and plots the Dice landscape across all weight combos
# ============================================================

# Paste the sweep table from ensemble_3model.py output here
# Format: (w_monai, w_dynunet, w_2d, threshold, val_dice)
SWEEP_RESULTS = [
    (1.00, 0.00, 0.00, 0.20, 0.6490), (1.00, 0.00, 0.00, 0.35, 0.6629), (1.00, 0.00, 0.00, 0.50, 0.6730),
    (0.00, 1.00, 0.00, 0.20, 0.7026), (0.00, 1.00, 0.00, 0.35, 0.7065), (0.00, 1.00, 0.00, 0.50, 0.7096),
    (0.00, 0.00, 1.00, 0.20, 0.3744), (0.00, 0.00, 1.00, 0.35, 0.3762), (0.00, 0.00, 1.00, 0.50, 0.3773),
    (0.50, 0.50, 0.00, 0.20, 0.6591), (0.50, 0.50, 0.00, 0.35, 0.6780), (0.50, 0.50, 0.00, 0.50, 0.7112),
    (0.45, 0.45, 0.10, 0.20, 0.6611), (0.45, 0.45, 0.10, 0.35, 0.6819), (0.45, 0.45, 0.10, 0.50, 0.7154),
    (0.40, 0.40, 0.20, 0.20, 0.6384), (0.40, 0.40, 0.20, 0.35, 0.6880), (0.40, 0.40, 0.20, 0.50, 0.7161),
    (0.38, 0.37, 0.25, 0.20, 0.6255), (0.38, 0.37, 0.25, 0.35, 0.6918), (0.38, 0.37, 0.25, 0.50, 0.7164),
    (0.35, 0.35, 0.30, 0.20, 0.6238), (0.35, 0.35, 0.30, 0.35, 0.7039), (0.35, 0.35, 0.30, 0.50, 0.7167),
    (0.35, 0.40, 0.25, 0.20, 0.6271), (0.35, 0.40, 0.25, 0.35, 0.7030), (0.35, 0.40, 0.25, 0.50, 0.7168),
    (0.30, 0.45, 0.25, 0.20, 0.6305), (0.30, 0.45, 0.25, 0.35, 0.7062), (0.30, 0.45, 0.25, 0.50, 0.7175),
    (0.30, 0.40, 0.30, 0.20, 0.6272), (0.30, 0.40, 0.30, 0.35, 0.7060), (0.30, 0.40, 0.30, 0.50, 0.7175),
    (0.25, 0.60, 0.15, 0.20, 0.6766), (0.25, 0.60, 0.15, 0.35, 0.7072), (0.25, 0.60, 0.15, 0.50, 0.7130),
]

df_sweep = pd.DataFrame(SWEEP_RESULTS, columns=['w_monai','w_dyn','w_2d','thresh','val_dice'])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Ensemble weight sweep — val Dice across all combinations', fontsize=12, fontweight='bold')

# Plot 1: threshold 0.20
for thresh, ax, title in zip([0.20, 0.35, 0.50], axes, ['Threshold 0.20', 'Threshold 0.35', 'Threshold 0.50']):
    sub = df_sweep[df_sweep.thresh == thresh].copy()
    colors = ['#4CAF50' if r == sub.val_dice.max() else '#2196F3' for r in sub.val_dice]
    bars = ax.bar(range(len(sub)), sub.val_dice.values, color=colors, alpha=0.8)
    ax.axhline(BASELINE_2D, color='red', lw=1.5, linestyle='--', alpha=0.7, label=f'2D baseline ({BASELINE_2D})')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Val Dice')
    ax.set_xlabel('Weight combo index')
    ax.set_ylim(0.3, 0.78)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    best_idx = sub.val_dice.idxmax() - sub.index[0]
    best_row = sub.loc[sub.val_dice.idxmax()]
    ax.annotate(f'Best: {best_row.val_dice:.4f}\nM={best_row.w_monai:.2f} D={best_row.w_dyn:.2f} 2={best_row.w_2d:.2f}',
                xy=(best_idx, best_row.val_dice), xytext=(best_idx+0.5, best_row.val_dice-0.04),
                fontsize=7, color='green', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='green', lw=0.8))

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'outputs', 'ensemble_weight_sweep.png'), dpi=120, bbox_inches='tight')
plt.show()

# Summary table
print('Top 5 weight combinations:')
print(df_sweep.sort_values('val_dice', ascending=False).head(5).to_string(index=False))
print(f'\nBest: MONAI={BEST_W_MONAI}  DynUNet={BEST_W_DYNUNET}  2D={BEST_W_2D}  thresh={BEST_THRESHOLD}  Val Dice=0.7175')

In [ ]:
# ============================================================
# CELL 3 — Test set inference
# Runs all 3 models on 36 test patients, ensembles probability
# maps, saves predictions. Caches per-patient prob maps.
# ============================================================
import cv2
from monai.inferers import sliding_window_inference
from torch.amp import autocast
from torch.utils.data import DataLoader

SW_OVERLAP = 0.5

def predict_3d(model, image_tensor):
    with torch.no_grad():
        with autocast('cuda', dtype=torch.bfloat16):
            logits = sliding_window_inference(
                inputs=image_tensor, roi_size=PATCH_SIZE,
                sw_batch_size=2, predictor=model,
                overlap=SW_OVERLAP, mode='gaussian')
    out = logits
    if hasattr(out, 'as_tensor'):
        out = out.as_tensor()
    return torch.sigmoid(out.float()).squeeze().cpu().numpy().astype(np.float32)

def predict_2d_volume(volume_np, target_shape):
    _, D, H, W = volume_np.shape
    prob_slices = []
    for i in range(D):
        sl   = (volume_np[:, i] * 255).astype(np.uint8)
        chs  = [cv2.resize(sl[c], (256, 256)) for c in range(3)]
        img_t = torch.tensor(
            np.transpose(np.stack(chs, axis=2).astype(np.float32)/255, (2,0,1))
        ).unsqueeze(0).float().to(device)
        with torch.no_grad():
            prob = torch.sigmoid(model_2d(img_t)).squeeze().cpu().numpy()
        prob_slices.append(cv2.resize(prob, (W, H)))
    pv = np.stack(prob_slices, axis=0)
    if pv.shape != target_shape:
        rv = np.zeros(target_shape, dtype=np.float32)
        for i in range(target_shape[0]):
            si = int(i * D / target_shape[0])
            rv[i] = cv2.resize(pv[min(si, D-1)], (target_shape[2], target_shape[1]))
        return rv
    return pv.astype(np.float32)

def post_process(binary_mask, min_size=50):
    struct  = ndimage.generate_binary_structure(3, 1)
    closed  = ndimage.binary_closing(binary_mask, structure=struct, iterations=2)
    labeled, n = ndimage.label(closed)
    if n == 0:
        return closed.astype(np.uint8)
    cleaned = np.zeros_like(closed, dtype=np.uint8)
    for i in range(1, n+1):
        if (labeled == i).sum() >= min_size:
            cleaned[labeled == i] = 1
    return cleaned

def dice_3d(pred, gt):
    p, g = pred.astype(bool), gt.astype(bool)
    return float(2*np.logical_and(p,g).sum() / (p.sum()+g.sum()+1e-8))

def iou_3d(pred, gt):
    p, g = pred.astype(bool), gt.astype(bool)
    return float(np.logical_and(p,g).sum() / (np.logical_or(p,g).sum()+1e-8))

test_list   = _build_data_list(PREPROCESSED_ROOT, 'test')
test_ds     = NpyDataset(test_list, transform=get_val_transforms())
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0, pin_memory=False)

results        = []
prob_cache     = []  # for visualisation cells
PRED_OUT_DIR   = os.path.join(ENSEMBLE_OUT_DIR, 'predictions')
os.makedirs(PRED_OUT_DIR, exist_ok=True)

for idx, batch in enumerate(tqdm(test_loader, desc='Ensemble inference')):
    images     = batch['image'].to(device)
    labels     = batch['label']
    patient_id = test_list[idx]['patient_id']
    vol_np     = batch['image'].squeeze(0).numpy()

    prob_monai   = predict_3d(monai_model, images)
    prob_dyn     = predict_3d(dyn_model,   images)
    prob_2d      = predict_2d_volume(vol_np, prob_monai.shape)

    prob_ens     = BEST_W_MONAI*prob_monai + BEST_W_DYNUNET*prob_dyn + BEST_W_2D*prob_2d
    binary_mask  = post_process((prob_ens > BEST_THRESHOLD).astype(np.uint8))

    gt_np   = (labels.squeeze().numpy() > 0).astype(bool)
    pred_np = binary_mask.astype(bool)

    # Handle shape mismatch (new dataset different axis order)
    if pred_np.shape != gt_np.shape:
        min_d = min(pred_np.shape[0], gt_np.shape[0])
        min_h = min(pred_np.shape[1], gt_np.shape[1])
        min_w = min(pred_np.shape[2], gt_np.shape[2])
        pred_np = pred_np[:min_d, :min_h, :min_w]
        gt_np   = gt_np[:min_d, :min_h, :min_w]

    dc  = dice_3d(pred_np, gt_np)
    iou = iou_3d(pred_np,  gt_np)

    results.append({'patient_id': patient_id, 'dice': dc, 'iou': iou,
                    'gt_voxels': int(gt_np.sum()), 'pred_voxels': int(pred_np.sum()),
                    'has_tumor': bool(gt_np.sum()>0), 'detected': bool(pred_np.sum()>0),
                    'w_monai_contrib': float(prob_monai[gt_np].mean()) if gt_np.sum()>0 else 0,
                    'w_dyn_contrib':   float(prob_dyn[gt_np].mean())   if gt_np.sum()>0 else 0,
                    'w_2d_contrib':    float(prob_2d[gt_np].mean())    if gt_np.sum()>0 else 0,
                    })

    np.save(os.path.join(PRED_OUT_DIR, f'patient{patient_id}_ensemble.npy'), binary_mask)

    if idx < 36:
        prob_cache.append({'patient_id': patient_id, 'image': vol_np,
                           'gt': gt_np, 'pred': pred_np,
                           'prob_monai': prob_monai, 'prob_dyn': prob_dyn,
                           'prob_2d': prob_2d, 'prob_ens': prob_ens, 'dice': dc})

tumor_dices = [r['dice'] for r in results if r['has_tumor']]
missed      = sum(1 for r in results if r['has_tumor'] and not r['detected'])
print(f'\n  Ensemble test results:')
print(f'  Mean Dice      : {np.mean(tumor_dices):.4f}')
print(f'  Missed tumours : {missed}/{len(tumor_dices)}')
print(f'  Detection rate : {100*(1-missed/len(tumor_dices)):.1f}%')
print(f'  Best Dice      : {max(tumor_dices):.4f}')

In [ ]:
# ============================================================
# CELL 4 — GT vs prediction overlay
# Shows 4 probability maps: MONAI / DynUNet / 2D / Ensemble
# + GT boundary overlaid on P2 channel for 5 key slices
# ============================================================

def visualise_ensemble(cache_entry, save_path=None):
    pid      = cache_entry['patient_id']
    image    = cache_entry['image']        # (3, D, H, W)
    gt       = cache_entry['gt']           # (D, H, W) bool
    pred     = cache_entry['pred']         # ensemble binary
    p_monai  = cache_entry['prob_monai']
    p_dyn    = cache_entry['prob_dyn']
    p_2d     = cache_entry['prob_2d']
    p_ens    = cache_entry['prob_ens']
    dice     = cache_entry['dice']

    # Best slice = most GT voxels
    gt_per_slice = gt.sum(axis=(1,2))
    best_sl = int(np.argmax(gt_per_slice)) if gt_per_slice.max()>0 else image.shape[1]//2
    D = image.shape[1]
    slices = list(dict.fromkeys([max(0,best_sl-4), max(0,best_sl-2),
                                  best_sl, min(D-1,best_sl+2), min(D-1,best_sl+4)]))
    n_sl = len(slices)

    fig, axes = plt.subplots(5, n_sl, figsize=(n_sl*3.2, 16))
    fig.suptitle(
        f'Ensemble — Patient {pid} | Dice={dice:.4f} | '
        f'GT: {int(gt.sum()):,} vox | Pred: {int(pred.sum()):,} vox',
        fontsize=11, fontweight='bold')

    row_data  = [('P2 + GT/Pred', image[1], None),
                 ('MONAI prob',   image[1], p_monai),
                 ('DynUNet prob', image[1], p_dyn),
                 ('2D prob',      image[1], p_2d),
                 ('Ensemble prob',image[1], p_ens)]

    for row, (title, mri, prob) in enumerate(row_data):
        for col, sl in enumerate(slices):
            ax = axes[row, col]
            ax.imshow(mri[sl], cmap='gray', vmin=0, vmax=1)

            if prob is not None:
                ax.imshow(prob[sl], cmap='hot', alpha=0.45, vmin=0, vmax=1)
                if gt[sl].sum()>0:
                    ax.contour(gt[sl].astype(float), levels=[0.5], colors='cyan', linewidths=1.5)
            else:
                if gt[sl].sum()>0:
                    ax.contour(gt[sl].astype(float), levels=[0.5], colors='lime', linewidths=2)
                if pred[sl].sum()>0:
                    ax.contour(pred[sl].astype(float), levels=[0.5], colors='red', linewidths=1.5, linestyles='--')

            if col == 0:
                ax.set_ylabel(title, fontsize=8)
            ax.set_title(f'Sl {sl}', fontsize=7)
            ax.axis('off')

    handles = [
        mpatches.Patch(edgecolor='lime',  facecolor='none', label='GT (row 1)'),
        mpatches.Patch(edgecolor='red',   facecolor='none', label='Ensemble pred (row 1)', linestyle='--'),
        mpatches.Patch(edgecolor='cyan',  facecolor='none', label='GT (prob rows)'),
    ]
    fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=8)
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()
    plt.close()

sorted_cache = sorted(prob_cache, key=lambda x: x['dice'])
n = len(sorted_cache)
show_cases   = [
    (sorted_cache[-1],   'best'),
    (sorted_cache[n//2], 'median'),
    (sorted_cache[0],    'worst'),
]
for entry, label in show_cases:
    print(f'\n--- {label.capitalize()} prediction (Dice={entry["dice"]:.4f}, Patient {entry["patient_id"]}) ---')
    fn = os.path.join(PROJECT_ROOT, 'outputs', f'ensemble_patient{entry["patient_id"]}_{label}.png')
    visualise_ensemble(entry, save_path=fn)
    print(f'Saved: {fn}')

In [ ]:
# ============================================================
# CELL 5 — 3D MIP rendering: GT vs ensemble prediction
# Max-intensity projection across axial/coronal/sagittal
# ============================================================

def plot_mip_ensemble(cache_entry):
    pid    = cache_entry['patient_id']
    gt     = cache_entry['gt'].astype(float)
    pred   = cache_entry['pred'].astype(float)
    img    = cache_entry['image'][1]   # P2 channel
    p_ens  = cache_entry['prob_ens']
    dice   = cache_entry['dice']

    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    fig.suptitle(f'3D MIP — Patient {pid} | Ensemble Dice = {dice:.4f}',
                 fontsize=11, fontweight='bold')

    axis_names = ['Axial (D)', 'Coronal (H)', 'Sagittal (W)']
    row_labels = ['GT', 'Ensemble Prediction', 'Ensemble Probability']

    for col, axis in enumerate([0, 1, 2]):
        mip_img  = img.max(axis=axis)
        mip_gt   = gt.max(axis=axis)
        mip_pred = pred.max(axis=axis)
        mip_prob = p_ens.max(axis=axis)

        for row, (overlay, cmap, alpha) in enumerate([
            (mip_gt,   'Greens', 0.55),
            (mip_pred, 'Reds',   0.55),
            (mip_prob, 'hot',    0.55),
        ]):
            ax = axes[row, col]
            ax.imshow(mip_img, cmap='gray')
            if overlay.max() > 0:
                ax.imshow(overlay, cmap=cmap, alpha=alpha, vmin=0, vmax=1)
            ax.axis('off')
            if col == 0:
                ax.set_ylabel(row_labels[row], fontsize=9)
            if row == 0:
                ax.set_title(axis_names[col], fontsize=9)

    plt.tight_layout()
    fn = os.path.join(PROJECT_ROOT, 'outputs', f'ensemble_mip_patient{pid}.png')
    plt.savefig(fn, dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fn}')

for entry, label in show_cases:
    print(f'\n--- MIP: {label.capitalize()} prediction ---')
    plot_mip_ensemble(entry)

In [ ]:
# ============================================================
# CELL 6 — Per-patient evaluation table
# Colour coded: green ≥0.7 / yellow ≥0.5 / orange ≥0.3 / red <0.3
# ============================================================

df = pd.DataFrame(results)
df['dice']     = df['dice'].round(4)
df['iou']      = df['iou'].round(4)
df['detected'] = df['detected'].map({True: 'Yes', False: 'No'})

df_display = df[['patient_id','dice','iou','gt_voxels','pred_voxels','detected']].copy()
df_display = df_display.sort_values('dice', ascending=False).reset_index(drop=True)

def color_dice(val):
    if val >= 0.7:   return 'background-color: #c8e6c9'
    elif val >= 0.5: return 'background-color: #fff9c4'
    elif val >= 0.3: return 'background-color: #ffe0b2'
    else:            return 'background-color: #ffcdd2'

styled = df_display.style.applymap(color_dice, subset=['dice'])
display(styled)

csv_path = os.path.join(ENSEMBLE_OUT_DIR, 'ensemble_per_patient.csv')
df_display.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')

tumor_dices = [r['dice'] for r in results if r['has_tumor']]
print(f'\nSummary:')
print(f'  Mean Dice   : {np.mean(tumor_dices):.4f}')
print(f'  Std Dice    : {np.std(tumor_dices):.4f}')
print(f'  Median Dice : {np.median(tumor_dices):.4f}')
print(f'  Best Dice   : {max(tumor_dices):.4f}')
print(f'  Worst Dice  : {min(tumor_dices):.4f}')
print(f'  Missed      : {sum(1 for r in results if r["has_tumor"] and not r["detected"])}')

In [ ]:
# ============================================================
# CELL 7 — Model contribution analysis
# Shows which model contributes most for each patient
# and where each individual model succeeds vs fails
# ============================================================

# Per-patient individual model Dice (from the per-patient CSVs)
# Update these from your evaluate_3d.py and evaluate_dynunet.py outputs
MONAI_PER_PATIENT = {
    '67': 0.9422, '78': 0.9028, '20': 0.8635, '66': 0.8560, '15': 0.8477,
    '177': 0.8440,'42': 0.8358, '54': 0.8330, '233': 0.8209,'48': 0.8130,
    '102': 0.8058,'106': 0.7990,'13': 0.7878, '23': 0.7820, '150': 0.7679,
    '120': 0.7667,'159': 0.7480,'149': 0.7210,'37': 0.7210, '174': 0.7050,
    '9': 0.6763,  '200': 0.6720,'65': 0.6660, '145': 0.6598,'36': 0.6440,
    '162': 0.6323,'123': 0.6277,'188': 0.5990,'14': 0.5544, '179': 0.5260,
    '186': 0.3165,'160': 0.2600,'194': 0.2388,'114': 0.2328,'193': 0.0,'204': 0.0
}
DYNUNET_PER_PATIENT = {
    '67': 0.9580, '66': 0.9180, '78': 0.9090, '15': 0.8680, '20': 0.8540,
    '54': 0.8318, '42': 0.8160, '106': 0.8024,'23': 0.7684, '14': 0.7390,
    '65': 0.7030, '13': 0.6848, '48': 0.6810, '159': 0.6590,'9': 0.6138,
    '36': 0.6030, '123': 0.5894,'150': 0.5890,'177': 0.5884,'149': 0.5880,
    '120': 0.5850,'102': 0.5567,'200': 0.5450,'233': 0.4430,'162': 0.4295,
    '194': 0.4130,'174': 0.3520,'145': 0.3268,'179': 0.2560,
    '37': 0.0,'114': 0.0,'160': 0.0,'193': 0.0,'188': 0.0,'186': 0.0,'204': 0.0
}

df_contrib = pd.DataFrame(results)[['patient_id','dice']].copy()
df_contrib.columns = ['patient_id', 'ensemble_dice']
df_contrib['monai_dice']   = df_contrib['patient_id'].map(lambda x: MONAI_PER_PATIENT.get(str(x), 0))
df_contrib['dynunet_dice'] = df_contrib['patient_id'].map(lambda x: DYNUNET_PER_PATIENT.get(str(x), 0))
df_contrib = df_contrib.sort_values('ensemble_dice', ascending=False)

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Model contribution — ensemble vs individual models', fontsize=12, fontweight='bold')

x    = np.arange(len(df_contrib))
pids = ['P'+str(p) for p in df_contrib.patient_id]

ax = axes[0]
ax.plot(x, df_contrib.ensemble_dice.values, 'o-', color='#9C27B0', lw=2, ms=5, label='Ensemble', zorder=3)
ax.plot(x, df_contrib.monai_dice.values,   's--', color='#2196F3', lw=1.5, ms=4, label='MONAI', alpha=0.8)
ax.plot(x, df_contrib.dynunet_dice.values, '^--', color='#4CAF50', lw=1.5, ms=4, label='DynUNet', alpha=0.8)
ax.axhline(BASELINE_2D, color='gray', lw=1.5, linestyle=':', label=f'2D baseline ({BASELINE_2D})', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(pids, fontsize=7, rotation=60)
ax.set_ylabel('Dice')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_title('Per-patient Dice: ensemble vs individual models (sorted by ensemble Dice)')

# Gain plot: ensemble - best(monai, dynunet)
ax2 = axes[1]
best_individual = np.maximum(df_contrib.monai_dice.values, df_contrib.dynunet_dice.values)
gain = df_contrib.ensemble_dice.values - best_individual
colors_gain = ['#4CAF50' if g >= 0 else '#F44336' for g in gain]
ax2.bar(x, gain, color=colors_gain, alpha=0.8)
ax2.axhline(0, color='black', lw=1)
ax2.set_xticks(x)
ax2.set_xticklabels(pids, fontsize=7, rotation=60)
ax2.set_ylabel('Ensemble gain vs best individual model')
ax2.grid(axis='y', alpha=0.3)
ax2.set_title('Ensemble gain over best single model (green = helped, red = hurt)')

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'outputs', 'ensemble_contribution_analysis.png'), dpi=120, bbox_inches='tight')
plt.show()

helped = (gain > 0.01).sum()
hurt   = (gain < -0.01).sum()
print(f'Ensemble improved on best individual: {helped}/{len(gain)} patients')
print(f'Ensemble worse than best individual : {hurt}/{len(gain)} patients')
print(f'Mean gain : {gain.mean():.4f}')

In [ ]:
# ============================================================
# CELL 8 — 2D baseline vs MONAI vs DynUNet vs Ensemble
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model comparison — all methods', fontsize=13, fontweight='bold')

# Left: mean Dice bar chart
ax = axes[0]
models_names = ['2D baseline\n(ResNet34)', 'MONAI\n3D UNet', 'DynUNet\n3D', 'Ensemble\n(3-model)']
dices  = [BASELINE_2D, MONAI_TEST, DYNUNET_TEST, ENSEMBLE_TEST]
colors = ['#78909C', '#2196F3', '#4CAF50', '#9C27B0']
bars   = ax.bar(models_names, dices, color=colors, alpha=0.85, width=0.55)
for bar, val in zip(bars, dices):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.008,
            f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean test Dice', fontsize=11)
ax.set_ylim(0, 1.0)
ax.set_title('Mean test Dice — all models')
ax.grid(axis='y', alpha=0.3)

# Right: scatter — ensemble vs MONAI per patient
ax2 = axes[1]
m_vals = [MONAI_PER_PATIENT.get(str(r['patient_id']), 0) for r in results]
e_vals = [r['dice'] for r in results]

ax2.scatter(m_vals, e_vals, c='#9C27B0', s=45, alpha=0.75, zorder=3)
# Diagonal = no change
ax2.plot([0,1],[0,1], 'k--', lw=1, alpha=0.4, label='No change')
ax2.fill_between([0,1],[0,1],[1,1], alpha=0.05, color='green')
ax2.fill_between([0,1],[0,0],[0,1], alpha=0.05, color='red')
ax2.text(0.7, 0.9, 'Ensemble\nbetter', fontsize=9, color='green', alpha=0.7)
ax2.text(0.7, 0.2, 'Ensemble\nworse',  fontsize=9, color='red',   alpha=0.7)
ax2.set_xlabel('MONAI Dice per patient')
ax2.set_ylabel('Ensemble Dice per patient')
ax2.set_title('Ensemble vs MONAI — per patient scatter')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'outputs', 'ensemble_model_comparison.png'), dpi=120, bbox_inches='tight')
plt.show()

print('='*55)
print('  Model comparison summary')
print('='*55)
for name, d in zip(models_names, dices):
    marker = ' ← BEST' if d == max(dices) else ''
    print(f'  {name.replace(chr(10)," "):20s} : {d:.4f}{marker}')
print(f'  Gap to 2D baseline : {ENSEMBLE_TEST-BASELINE_2D:+.4f}')

In [ ]:
# ============================================================
# CELL 9 — Full summary card
# ============================================================

tumor_dices  = [r['dice'] for r in results if r['has_tumor']]
mean_ens     = float(np.mean(tumor_dices))
best_ens     = float(max(tumor_dices))
worst_ens    = float(min(tumor_dices))
det_rate     = 100*sum(r['detected'] for r in results if r['has_tumor'])/len(tumor_dices)
missed       = sum(1 for r in results if r['has_tumor'] and not r['detected'])

fig, ax = plt.subplots(figsize=(14, 7))
ax.axis('off')
fig.patch.set_facecolor('#1a1a2e')

ax.text(0.5, 0.97, 'Combination C Ensemble — Final Result Summary',
        ha='center', va='top', transform=ax.transAxes,
        fontsize=14, fontweight='bold', color='white')

config_info = [
    ('Models',        'MONAI ResNet UNet 3D + DynUNet 3D + 2D ResNet34'),
    ('Weight MONAI',  f'{BEST_W_MONAI}'),
    ('Weight DynUNet',f'{BEST_W_DYNUNET}'),
    ('Weight 2D',     f'{BEST_W_2D}'),
    ('Threshold',     f'{BEST_THRESHOLD}'),
    ('Val Dice',      '0.7175 (weight sweep)'),
    ('MONAI params',  '12,865,085'),
    ('DynUNet params','16,665,443'),
    ('Patch size',    str(PATCH_SIZE)),
    ('GPU',           gpu_name),
]
result_info = [
    ('Ensemble Dice (mean)',  f'{mean_ens:.4f}'),
    ('Ensemble Dice (best)',  f'{best_ens:.4f}'),
    ('Ensemble Dice (worst)', f'{worst_ens:.4f}'),
    ('Detection rate',        f'{det_rate:.1f}%'),
    ('Missed tumours',        f'{missed} / {len(tumor_dices)}'),
    ('MONAI test Dice',       f'{MONAI_TEST:.4f}'),
    ('DynUNet test Dice',     f'{DYNUNET_TEST:.4f}'),
    ('2D baseline',           f'{BASELINE_2D}'),
    ('Gain vs 2D baseline',   f'{mean_ens-BASELINE_2D:+.4f}'),
    ('HD95 (mm)',             '25.49'),
]

def draw_box(ax, title, items, x0, y0, bg, tc):
    rect = mpatches.FancyBboxPatch((x0-0.02, y0-0.03), 0.44, 0.02+len(items)*0.071,
        boxstyle='round,pad=0.01', facecolor=bg, edgecolor='#444',
        linewidth=1.5, transform=ax.transAxes, zorder=0)
    ax.add_patch(rect)
    ax.text(x0+0.20, y0+len(items)*0.071-0.01, title,
            ha='center', va='bottom', transform=ax.transAxes,
            fontsize=11, fontweight='bold', color=tc)
    for i, (label, val) in enumerate(items):
        y = y0 + (len(items)-1-i)*0.071
        ax.text(x0+0.01, y, label+' :', ha='left', va='center',
                transform=ax.transAxes, fontsize=9, color='#aaa')
        ax.text(x0+0.41, y, str(val), ha='right', va='center',
                transform=ax.transAxes, fontsize=9, color='white', fontweight='bold')

draw_box(ax, 'Ensemble Config', config_info, 0.02, 0.04, '#16213e', '#64b5f6')
draw_box(ax, 'Test Set Results', result_info, 0.54, 0.04, '#0f3460', '#a5d6a7')

status_color = '#1b5e20' if mean_ens >= BASELINE_2D else '#b71c1c'
status_text  = ('Ensemble surpassed 2D baseline' if mean_ens >= BASELINE_2D
                else f'Gap to 2D baseline: {mean_ens-BASELINE_2D:+.4f} — increase MONAI Dice to close gap')
ax.text(0.5, 0.015, status_text, ha='center', va='bottom', transform=ax.transAxes,
        fontsize=10, fontweight='bold', color='white',
        bbox=dict(boxstyle='round,pad=0.4', facecolor=status_color, alpha=0.9))

fn = os.path.join(PROJECT_ROOT, 'outputs', 'ensemble_summary_card.png')
plt.savefig(fn, dpi=130, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print(f'Saved: {fn}')
print('\n' + '='*55)
print('  Ensemble pipeline complete')
print('='*55)
print(f'  Ensemble Dice : {mean_ens:.4f}')
print(f'  2D baseline   : {BASELINE_2D}')
print(f'  Detection rate: {det_rate:.1f}%')
print(f'  Next step: retrain MONAI → increase ensemble Dice')